# Load dynamic data sources

This notebook keeps the original loader workflow (price, generation, LGC, tariff, physical constraints, then dataset assembly) and adds only the requested CSV-backed asset/configuration layer and A–F decision routing. `M` identifies a multi-series source.

| # | Single series | Multi-series |
|---:|---|---|
| 1 | `A1_historical_price` | `A1M_historical_price_resampled_<state>` |
| 2 | `A2_future_price_ST_predicted` | `A2M_future_price_ST_predicted_resampled_<state>` |
| 3 | `A3_future_price_LT_predicted` | `A3M_future_price_LT_predicted_resampled_<state>` |
| 4 | `B1_historical_generation` | `B1M_historical_generation_resampled_<asset>` |
|   | `B1_historical_generation_standard_year` | `B1M_historical_generation_standard_year_resampled_<asset>` |
| 5 | `B2_future_generation_ST_predicted` | `B2M_future_generation_ST_predicted_resampled_<asset>` |
| 6 | `B3_future_generation_LT_predicted` | `B3M_future_generation_LT_predicted_resampled_<asset>` |
| 7 | `C1_LGC` | |
| 8 | `D1_network_time_of_use_tarrif_mapping` | |
|   | `D2_network_tarrif_values` | |
| 9 | `E1_physical_constraints_mapping` | |
| 10 | `F1_BESS_size` | `F1M_BESS_size_resampled_<asset>` |


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'optimizer' / 'configuration.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the repository or one of its subdirectories.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from optimizer.configuration import ConfigurationEditor, CsvTableStore, DynamicDataLoader

store = CsvTableStore()


## Asset and configuration tables

Use the two tabs below to create or edit records. Choice fields are populated from the lookup CSVs, and dependent choices follow the supplied process maps. Mark exactly one configuration as active.


In [ ]:
editor = ConfigurationEditor(store).display()


## Review the active routing

The inventory shows the exact current-standard filename for every source and whether it exists. Missing unselected alternatives are allowed; every selected source must exist when the loader runs.


In [3]:
loader = DynamicDataLoader(store=store)
display(store.assets())
display(store.configurations())
display(loader.inventory())


,asset_name,state,tariff_code,demand_charge_off_peak,demand_charge_shoulder,demand_charge_peak,export_charge_sun_soaker,project_poi_export_limit_MW,project_poi_import_limit_MW,project_poi_export_limit_MWh_per_interval,...,LGC_price_override_per_MWh,BESS_technology,BESS_coupling_type,BESS_duration_hours,BESS_power_MW,BESS_usable_fraction,BESS_energy_override_MWh,BESS_Start_SoC,BESS_cycles_per_day,BESS_lifetime_years
0,Orange 2B,NSW,BHND4LS,3432.106344,10295.563478,11497.084032,918.470947,4.99,4.2,4.99,...,0.000000,Jinko,AC,2.0,4.96,0.945763,0.000000,0.0,1.0,20
1,Punch's Creek,QLD,BHND4LS,3432.106344,10295.563478,11497.084032,918.470947,400.00,400.0,400.00,...,17.400849,CATL,DC,4.0,400.00,1.000000,1738.570182,0.0,1.0,20


,configuration_name,active,asset,execution_mode,display_window_scheduler_visual,optimization_start_date,optimization_end_date,optimization_granularity_in_minutes,dataset_date_range,dataset_year_month_mapping,...,generation_profile,generation_run_synthetic_distribution,generation_series_start,generation_series_stop,bess_run_synthetic_distribution,bess_series_start,bess_series_stop,price_source,generation_source,bess_source
0,Default,False,Orange 2B,sequential,True,01/01/2025 00:00,31/12/2025 23:55,60,2025-01-01T00:00:00|2025-12-31T23:55:00|60min,2025-01;2025-02;2025-03;2025-04;2025-05;2025-0...,...,Standard year,False,0,100,False,0,10,A1,B1S,F1
1,Punch's Creek,False,Punch's Creek,sequential,True,01/01/2025 00:00,31/12/2025 23:55,30,2025-01-01T00:00:00|2025-12-31T23:55:00|30min,2025-01;2025-02;2025-03;2025-04;2025-05;2025-0...,...,Standard year,False,0,100,False,0,10,A3,B1S,F1
2,Orange 2B,False,Orange 2B,sequential,True,01/01/2022 00:00,31/12/2024 23:55,30,2022-01-01T00:00:00|2024-12-31T23:55:00|30min,2022-01;2022-02;2022-03;2022-04;2022-05;2022-0...,...,Standard year,False,0,100,False,0,10,A1,B1S,F1
3,Orange 2B config,False,Orange 2B,sequential,True,01/01/2022 00:00,31/12/2024 23:55,5,2022-01-01T00:00:00|2024-12-31T23:55:00|5min,2022-01;2022-02;2022-03;2022-04;2022-05;2022-0...,...,Standard year,False,0,100,False,0,10,A1,B1S,F1
4,Punch's Creek config,True,Punch's Creek,sequential,True,01/01/2025 00:00,31/12/2025 23:55,30,2025-01-01T00:00:00|2025-12-31T23:55:00|30min,2025-01;2025-02;2025-03;2025-04;2025-05;2025-0...,...,Standard year,False,0,100,False,0,10,A1,B1S,F1


,Code,Selected,Status,File
0,A1,True,available,A1_historical_price.csv
1,A1M,False,missing,A1M_historical_price_resampled_qld.csv
2,A2,False,missing,A2_future_price_ST_predicted.csv
3,A2M,False,missing,A2M_future_price_ST_predicted_resampled_qld.csv
4,A3,False,available,A3_future_price_LT_predicted.csv
5,A3M,False,missing,A3M_future_price_LT_predicted_resampled_qld.csv
6,B1,False,available,B1_historical_generation.csv
7,B1M,False,available,B1M_historical_generation_resampled_Punch's_Cr...
8,B1S,True,available,B1_historical_generation_standard_year.csv
9,B1SM,False,missing,B1M_historical_generation_standard_year_resamp...


## Build optimizer inputs

Run this after saving the active configuration. It creates taxonomy-named source tables in `2_Processed_data` and rebuilds `Dataset/dataset.csv`. The optimizer reads those files and the active CSV configuration directly.


In [4]:
sources = loader.load_selected_sources(save=True)
display({
    'dataset': sources.dataset.shape,
    'price_actual': sources.price_actual.shape,
    'price_predicted': sources.price_predicted.shape,
    'generation': sources.generation.shape,
    'bess_sizes': sources.bess_sizes.shape,
})


{'dataset': (17520, 10),
 'price_actual': (17520, 2),
 'price_predicted': (17520, 2),
 'generation': (17520, 2),
 'bess_sizes': (1, 4)}